# 02. Running sums

**Tier:** Deep. **Why you'd use this:** A subarray is a contiguous run. At each value, the best run ending here is either this value alone, or this value added to the previous run.

**After this lesson you can:** build a running total array; answer a "sum so far" question in one pass.
About 20 minutes. **Uses:** [Pivot](01-pivot-lesson.ipynb)


## Quick review

1. How is a running sum different from a single total?
2. Name one question a running-sum array answers quickly.

<details><summary>Answers</summary>

For 02 running sum lesson: skim [Pivot](01-pivot-lesson.ipynb) if needed.
</details>


## Recognize it

You need every prefix total: the sum of the first value, the first two, the first three, and so on. The prompt says "running sum", "prefix sums", or "sum so far at each index".

Signals: one forward accumulate, and later answers read those prefixes. A look-alike that is not this pattern: maximum subarray — that chooses whether to extend or restart, and returns one number, not the prefix list.

A second look-alike: a single final total. If the caller only prints `sum(nums)` once, building an `n`-length prefix list wastes memory for no later query. Build the running list when those intermediate totals are the answer or the preprocess for what comes next.


## Why it works this way

The sum of the first `i + 1` values equals the sum of the first `i` values plus `nums[i]`. That recurrence means each prefix reuses the previous prefix. Building the whole list is one pass: start at `0`, add the next value, append. Time grows with `n`, and the answer needs `n` slots of memory for the totals you return.

Without reuse you would call `sum(nums[:i+1])` for each `i`. Each call rescans from the start, so total work grows like `1 + 2 + … + n`, which is proportional to `n²`. The running total removes that nested scan. See [Lists](../../internals/03-lists/01-dynamic-array-lesson.ipynb) for why appending `n` times into a result list stays amortized linear under CPython's overallocation strategy for dynamic arrays.

Prefix arrays are the preprocess behind range sums: `sum(nums[L:R+1])` becomes `prefix[R+1] - prefix[L]` when `prefix[0] = 0` and `prefix[i]` stores the sum of the first `i` values. This lesson returns the inclusive form where `answer[i]` is the sum through index `i`, which is what "running sum" usually means in prompts. Converting between the two shapes is a shift by one plus a leading zero on the exclusive form.

Kadane's maximum-subarray pass also walks with a running value, but it may reset when the running total hurts the answer. A pure running sum never resets; every value joins the prefix. Keep those problems separate even though both say "running" in casual speech. If the prompt asks for one best sum, reach for Kadane. If it asks for every prefix, build the list shown below.

The design reason for storing every prefix, not only the final total, is future random access. Range queries, pivot checks, and difference arrays all read two prefixes and subtract. Paying linear memory once is cheaper than rescanning a slice on every later question. That is why timed challenges treat the running-sum array as the foundation of the prefix-sum family rather than as a one-off party trick.

When you mutate the input in place, `nums[i] += nums[i - 1]` for `i` from `1` onward, you still compute the same inclusive prefixes but overwrite the caller's list. Prefer a fresh list in teaching code so the original values stay available for checks; prefer in-place only when the API allows destroying the input and memory is tight.


## When to use it

Use a running-sum array when you need every prefix total, or as the first step toward many range-sum answers. Use Kadane when you need the best contiguous sum, not the prefixes. Use a single `sum` when you only need the final total once. Prefer `itertools.accumulate` when you want the same idea in the standard library.

Prefer an exclusive prefix with a leading zero when the next lesson will answer many `[L, R]` queries by subtraction. Prefer the inclusive form when the prompt literally asks for the running sum list. Convert between them with one shift rather than rewriting the scan.


## Brute force, and why it's slow

Resum each prefix from the start. Count the additions.


In [1]:
def running_sum_slow(nums):
    additions = 0
    answer = []
    for end in range(len(nums)):
        total = 0
        for index in range(end + 1):
            total += nums[index]
            additions += 1
        answer.append(total)
    return answer, additions


print(running_sum_slow([1, -3, 4]))
print(running_sum_slow(list(range(30)))[1])

([1, -2, 2], 6)
465


## Key insight

Each prefix is the previous prefix plus the next value. Append that total and move on — never restart the sum from index `0`. The inclusive answer at index `i` is exactly the sum through `i`.


## Worked example

Return the running sums: each position holds the sum of every value up to that index.


In [2]:
def running_sum(nums: list[int]) -> list[int]:
    # 1. Grow one total; append after each add.
    totals = []
    current = 0
    for value in nums:
        current += value
        totals.append(current)
    return totals


print(running_sum([1, -3, 4]))
print(running_sum([3, 1, 2, 10, 1]))

[1, -2, 2]
[3, 4, 6, 16, 17]


## Trace

`[1, -3, 4]`:

| value | current after add | totals |
|---|---|---|
| 1 | 1 | `[1]` |
| -3 | -2 | `[1, -2]` |
| 4 | 2 | `[1, -2, 2]` |


## Cost

One add and one append per value: time and output memory both grow with `n`. The rescan version pays about `n² / 2` additions for the same output. In-place updates drop the extra list but destroy the input; exclusive prefixes need `n + 1` slots including the leading zero.


## Variants

Not solved here: [Maximum Subarray](../../challenges/problems/21-maximum-subarray.ipynb), [Subarray Sum Equals K](../../challenges/problems/43-subarray-sum-equals-k.ipynb), [Product of Array Except Self](../../challenges/problems/06-product-of-array-except-self.ipynb).


## Common mistake

This bug is silent. The function appends `value` instead of the running total, so the list is a copy of the input. There is no traceback. The print is `[1, -3, 4]`, and the answer is `[1, -2, 2]`.


In [3]:
def running_sum_wrong(nums):
    totals = []
    current = 0
    for value in nums:
        current += value
        totals.append(value)  # forgot the cumulative total
    return totals


print(running_sum_wrong([1, -3, 4]))

[1, -3, 4]


The fix appends `current`, the sum so far. The buggy cell prints `[1, -3, 4]` for that input because it appends each raw value instead of the running total.


In [4]:
print(running_sum([1, -3, 4]))

[1, -2, 2]


## Python idioms

`itertools.accumulate` builds the same list. A manual loop makes the `current` update obvious when you are still learning the pattern.


In [5]:
from itertools import accumulate


def running_sum_lib(nums: list[int]) -> list[int]:
    return list(accumulate(nums))


print(running_sum_lib([1, -3, 4]))

[1, -2, 2]


## Recap

- Each prefix reuses the previous prefix.
- Append the cumulative total, not the raw value.
- Running sums are the preprocess behind range queries.

Next: [practice](02-running-sum-practice.ipynb).
